# 02 — Classical + XLNet baselines (Wang et al. Table 3 configs)

In [ ]:
!rm -rf /content/repo
!git clone https://github.com/kvarun314/pes-mtech-project.git /content/repo
%cd /content/repo
!pip install -q -e ".[colab]"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RUNS_DIR = "/content/drive/MyDrive/pes-mtech-project/runs"

## Load a fresh split of the Phase 1 Kaggle dataset
This is a standalone Colab runtime — `/content/` from notebook 01 does not
carry over. Get `kaggle.json` from https://www.kaggle.com/settings -> Create
New Token, upload it via the sidebar (lands in `/content/`), then run the next
two cells to fetch the dataset into this runtime. This is a fresh random split
of the same CSV the LoRA run trained on, not the identical train/test rows
(Phase 1's own split lives only inside that notebook's run) — the baselines
are compared against each other on this split, not against Phase 1's exact
held-out set.

In [ ]:
import os, shutil

os.makedirs("/root/.kaggle", exist_ok=True)
if os.path.exists("/content/kaggle.json"):
    shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!pip install -q kaggle
!kaggle datasets download -d arhamrumi/amazon-product-reviews -p /content/data --unzip

In [ ]:
import pandas as pd

df = pd.read_csv("/content/data/Reviews.csv").dropna(subset=["Text", "Score"])
# Cap the working set: SVM with a non-linear kernel is roughly quadratic in
# row count, so the full ~568k-row CSV is infeasible on a Colab GPU/CPU within
# a reasonable session. 20,000 matches the XLNet cell below for a fair,
# comparable-scale baseline table.
df = df.sample(n=min(len(df), 20000), random_state=42)
texts, labels = df["Text"].tolist(), df["Score"].astype(int).tolist()

In [ ]:
from sklearn.model_selection import train_test_split
from agentic_sentiment.baselines.classical import (
    train_decision_tree, train_svm, train_naive_bayes, evaluate,
)

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)
results = {}
for name, trainer in [("decision_tree", train_decision_tree), ("svm_sigmoid", train_svm),
                      ("naive_bayes", train_naive_bayes)]:
    pipeline = trainer(X_train, y_train)
    results[name] = evaluate(pipeline, X_test, y_test)
    print(name, results[name])

import json, os
os.makedirs("/content/drive/MyDrive/pes-mtech-project", exist_ok=True)
with open("/content/drive/MyDrive/pes-mtech-project/results_run2_baselines.json", "w") as f:
    json.dump(results, f, indent=2)
# Persisted now so a later XLNet disconnect doesn't lose these (cheap to recompute, but why risk it)

In [ ]:
from agentic_sentiment.train.run_dir import RunDir
from agentic_sentiment.baselines.xlnet import run_training as run_xlnet

run_dir = RunDir(base_dir=DRIVE_RUNS_DIR, run_id="xlnet")
# local_output_dir stays off Drive -- the Drive-side checkpoints/ copy
# (pruned to the 2 most recent) is already durable; staging HF's own
# unpruned checkpoint-* dirs there too would double Drive usage for
# nothing. A reconnect under this same run_id resumes automatically
# from the latest Drive-side checkpoint instead of restarting at step 0.
results["xlnet"] = run_xlnet(X_train, y_train, run_dir, local_output_dir="/content/xlnet_output")
print(results["xlnet"])

In [ ]:
import json
with open("/content/drive/MyDrive/pes-mtech-project/results_run2_baselines.json", "w") as f:
    json.dump(results, f, indent=2)